Machine Learning for Scientists: Topic 1 - Fundamentals of Learning & Uncertainty
This notebook explores the core foundational principles of machine learning applied to scientific problem-solving:   Supervised vs. Unsupervised Paradigms: Demonstrates how data structure dictates whether we map inputs to targets $P(Y \mid X)$ or explore data distributions $P(X)$.   Probabilistic Modeling: Quantifies empirical noise and parameter uncertainty using continuous probability distributions.   Estimation Frameworks: Implements and compares Maximum Likelihood Estimation (MLE) and Maximum A Posteriori Estimation (MAP).

In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt

# Set seed for reproducibility
np.random.seed(42)

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Libraries imported successfully.")

Step 1: Data Preparation & Task FormulationSummaryIn machine learning, tasks are categorized by how observations are paired:   Supervised Learning (Regression/Classification): Learns a predictive mapping from input features $X$ to observed continuous or discrete targets $Y$. The goal is to estimate $P(Y \mid X)$ to predict outcomes for unseen inputs.   Unsupervised Learning (Clustering/Pattern Discovery): Learns the underlying structure, relationships, or latent groupings within input features $X$ without provided labels. The goal is to model $P(X)$.   The code below generates synthetic datasets for both tasks and visualizes their structural differences.

In [ ]:
# 1. Supervised Learning Dataset: Regression (X -> Y)
n_samples = 100
X_sup = np.random.uniform(-3, 3, size=(n_samples, 1))
true_w = 2.5
true_b = 1.0
noise = np.random.normal(0, 1.2, size=(n_samples, 1))
Y_sup = true_w * X_sup + true_b + noise

# 2. Unsupervised Learning Dataset: Clustering (X only)
cluster_1 = np.random.normal(loc=[-2, -2], scale=0.8, size=(50, 2))
cluster_2 = np.random.normal(loc=[2, 2], scale=0.8, size=(50, 2))
X_unsup = np.vstack([cluster_1, cluster_2])

# Visualizing datasets
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(X_sup, Y_sup, color='blue', alpha=0.7)
axes[0].set_title("Supervised Learning Task: Regression (X, Y)")
axes[0].set_xlabel("Feature (X)")
axes[0].set_ylabel("Target Label (Y)")

axes[1].scatter(X_unsup[:, 0], X_unsup[:, 1], color='purple', alpha=0.7)
axes[1].set_title("Unsupervised Learning Task: Finding Structure (X)")
axes[1].set_xlabel("Feature X1")
axes[1].set_ylabel("Feature X2")

plt.tight_layout()
plt.show()

Step 2: Parameter Estimation Principles (MLE vs. MAP)SummaryTo fit probability models to scientific data, we optimize model parameters $\theta$ using two primary principles:   Maximum Likelihood Estimation (MLE): Selects parameter values that maximize the likelihood of the observed data:
   $$\hat{\theta}_{\mathrm{MLE}} = \arg\max_\theta P(D \mid \theta)$$   
Key Characteristic: Relies exclusively on observed evidence. When sample sizes are small, it can be sensitive to noisy measurements.   Maximum A Posteriori Estimation (MAP): Incorporates prior beliefs $P(\theta)$ about parameter distributions alongside data likelihood:
   $$\hat{\theta}_{\mathrm{MAP}} = \arg\max_\theta P(D \mid \theta) P(\theta)$$   
Key Characteristic: Acts as a regularizer, penalizing extreme parameter values when data is limited.   The code below generates $N=10$ samples from a Gaussian distribution $D \sim \mathcal{N}(\mu^*=5.0, \sigma^2=4.0)$ and computes closed-form estimates for MLE and MAP with a prior centered at $\mu=0$.   

In [ ]:
# True underlying data parameters
true_mu = 5.0
data_sigma = 2.0
N = 10  # Small sample size to emphasize the impact of prior in MAP

# Generate synthetic observations
data = np.random.normal(loc=true_mu, scale=data_sigma, size=N)

# Define MAP Prior distribution over mu: N(prior_mu, prior_sigma^2)
prior_mu = 0.0
prior_sigma = 1.0

# 1. Analytical Solution for MLE of Gaussian Mean
mle_mu = np.mean(data)

# 2. Analytical Solution for MAP of Gaussian Mean with Gaussian Prior
# Precision-weighted average of data mean and prior mean
precision_data = N / (data_sigma ** 2)
precision_prior = 1 / (prior_sigma ** 2)

map_mu = (precision_data * mle_mu + precision_prior * prior_mu) / (precision_data + precision_prior)

print(f"Sample Size (N): {N}")
print(f"True Parameter (Mu): {true_mu}")
print(f"MLE Estimate: {mle_mu:.4f}")
print(f"MAP Estimate: {map_mu:.4f}")

Step 3: Visualizing Objective Landscapes Across Parameter SpaceSummaryThis step visualizes how the prior distribution pulls the MAP parameter estimate away from the purely empirical MLE peak:   Likelihood Curve $P(D \mid \mu)$: Peak corresponds to the standard empirical average ($\hat{\mu}_{\mathrm{MLE}}$).   Prior Curve $P(\mu)$: Represents baseline assumptions before observing data.   Posterior Curve $P(\mu \mid D)$: The product of Likelihood and Prior. The MAP estimate sits at the peak of this posterior curve, serving as a balance between prior beliefs and empirical evidence.

In [ ]:
# Candidate parameter range for mu
mu_grid = np.linspace(-2, 8, 500)

# 1. Log Likelihood
log_likelihood = np.sum([stats.norm.logpdf(data, loc=m, scale=data_sigma) for m in mu_grid], axis=1)

# 2. Log Prior
log_prior = stats.norm.logpdf(mu_grid, loc=prior_mu, scale=prior_sigma)

# 3. Log Posterior (Unnormalized MAP objective)
log_posterior = log_likelihood + log_prior

# Exponentiate and normalize distributions for visualization comparison
likelihood_norm = np.exp(log_likelihood - np.max(log_likelihood))
prior_norm = np.exp(log_prior - np.max(log_prior))
posterior_norm = np.exp(log_posterior - np.max(log_posterior))

# Plotting curves
plt.figure(figsize=(10, 6))
plt.plot(mu_grid, likelihood_norm, label='Likelihood P(D|μ) [MLE Peak]', color='green', linewidth=2)
plt.plot(mu_grid, prior_norm, label='Prior P(μ)', color='red', linestyle='--', linewidth=2)
plt.plot(mu_grid, posterior_norm, label='Posterior P(μ|D) [MAP Peak]', color='blue', linewidth=2.5)

plt.axvline(mle_mu, color='green', linestyle=':', label=f'MLE = {mle_mu:.2f}')
plt.axvline(map_mu, color='blue', linestyle=':', label=f'MAP = {map_mu:.2f}')
plt.axvline(true_mu, color='black', linestyle='-', alpha=0.6, label=f'True μ = {true_mu:.2f}')

plt.title('Parameter Estimation: Likelihood vs Prior vs Posterior', fontsize=14)
plt.xlabel('Parameter μ', fontsize=12)
plt.ylabel('Normalized Probability / Density', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True)
plt.show()

Step 4: Asymptotic Behavior & Sample Size ConvergenceSummaryThis experiment tracks parameter convergence as sample size $N$ increases from 1 to 150:Small Sample Sizes ($N < 20$): The prior heavily influences MAP estimation, keeping it closer to the prior mean ($\mu=0$) and mitigating overfitting to noisy small samples.   Large Sample Sizes ($N \to \infty$): Data evidence dominates the prior term ($P(D \mid \mu) \gg P(\mu)$). As a result, the MAP estimate converges to the MLE estimate, and both approach the ground truth parameter value $\mu^* = 5.0$.

In [ ]:
sample_sizes = np.arange(1, 151)
mle_estimates = []
map_estimates = []

# Generate large dataset stream
data_stream = np.random.normal(loc=true_mu, scale=data_sigma, size=150)

for n in sample_sizes:
    sub_data = data_stream[:n]
    mle = np.mean(sub_data)

    prec_data = n / (data_sigma ** 2)
    prec_prior = 1 / (prior_sigma ** 2)
    map_est = (prec_data * mle + prec_prior * prior_mu) / (prec_data + prec_prior)

    mle_estimates.append(mle)
    map_estimates.append(map_est)

plt.figure(figsize=(10, 5))
plt.plot(sample_sizes, mle_estimates, label='MLE Estimate', color='green', alpha=0.8)
plt.plot(sample_sizes, map_estimates, label='MAP Estimate', color='blue', alpha=0.8)
plt.axhline(true_mu, color='black', linestyle='--', label='True μ (5.0)')

plt.title('Convergence of MLE and MAP Estimates as Sample Size N Increases', fontsize=14)
plt.xlabel('Sample Size (N)', fontsize=12)
plt.ylabel('Estimated Value of μ', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True)
plt.show()